# Exercise 1: how much energy does a flare release? ★★☆☆☆

*Covers notebook 4.*

A flare's **equivalent duration** is the time the star would need, in
quiescence, to emit the energy the flare added:

$$
\mathrm{ED} = \int \left(\frac{F(t)}{F_\mathrm{q}} - 1\right) dt ,
$$

with $F_\mathrm{q}$ the quiescent flux. It needs no distance and no
calibration, only the flux ratio, so it is the usual first measure of a
flare's size. Multiplied by the star's quiescent luminosity in a band, it
gives the flare's energy in that band.

**Goal:** measure the equivalent duration of all four flares in β and in ρ,
and their ratio.

In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from astropy.visualization import AsinhStretch, ImageNormalize, ZScaleInterval

import argex
from argex import sky, times

warnings.filterwarnings("ignore", category=RuntimeWarning)  # all-NaN slices
GREEK = {"beta": "β", "rho": "ρ"}  # band names in plot labels
plt.rcParams.update({"figure.dpi": 110, "image.origin": "lower"})

# The staged sample. Set ARGUS_SAMPLE to read another copy, such as a download.
SAMPLE = Path(os.environ.get("ARGUS_SAMPLE", "~/s3/schmidt-observatory-system/argus/prototype_v1")).expanduser()
assert (SAMPLE / "manifest.json").is_file(), f"no sample at {SAMPLE}: set ARGUS_SAMPLE"

import json

from astropy.table import Table

In [ ]:
from astropy.stats import SigmaClip
from astropy.table import Table
from photutils.aperture import ApertureStats, CircularAnnulus, CircularAperture, aperture_photometry

EXPTIME = 60.0                                # s, every frame
SEGMENT_PIXEL, DETECTOR_PIXEL = 0.879, 1.012  # arcsec
R = 4 * DETECTOR_PIXEL / SEGMENT_PIXEL        # the zeropoints' 4.05" aperture, in segment pixels
ANNULUS = (R + 3, 15.5)                       # what fits in a 32-pixel cutout


def aperture_flux(cube, x, y, radius=R, annulus=ANNULUS):
    # Background-subtracted flux (e-/s) and its error in each plane of a cube,
    # with the background the sigma-clipped median of the annulus.
    aperture = CircularAperture((x, y), r=radius)
    ring = CircularAnnulus((x, y), r_in=annulus[0], r_out=annulus[1])
    clip = SigmaClip(sigma=3.0)
    flux, err = np.full(len(cube), np.nan), np.full(len(cube), np.nan)
    for k, plane in enumerate(np.asarray(cube, np.float64)):
        sky_ = ApertureStats(plane, ring, sigma_clip=clip)
        phot = aperture_photometry(plane - sky_.median, aperture,
                                   error=np.full(plane.shape, sky_.std), method="exact")
        flux[k] = phot["aperture_sum"][0]
        # the sky noise from the annulus, plus the source's photon noise
        err[k] = np.hypot(phot["aperture_sum_err"][0], np.sqrt(max(flux[k], 0) / EXPTIME))
    return flux, err


def light_curve(frames, epochs, flux, err, zp, flags=None):
    # A table of calibrated magnitudes; a frame is good when it is calibrated,
    # has a positive flux and (with DQ flags) none of argex.dq.BAD under the aperture.
    mag = zp - 2.5 * np.log10(flux)
    good = np.isfinite(mag)
    if flags is not None:
        good &= (flags & int(argex.dq.BAD)) == 0
    return Table({
        "epoch": epochs, "mjd": times.to_mjd(epochs),
        "frame_key": [f.frame_key for f in frames],
        "flux": flux, "flux_err": err, "zp": zp,
        "mag": mag, "mag_err": 2.5 / np.log(10) * err / flux,
        "dq": np.zeros(len(flux), np.uint32) if flags is None else flags, "good": good,
    })

## 1. The flux ratio of one flare

Write `flux_ratio(flare_id, band)` returning the minutes from the peak and
$F/F_\mathrm{q}$ for every frame of one flare's stack. Take the quiescent
flux as the median before the flare starts, and decide how to treat frames
the pointing left half-covered (NaN pixels in the aperture).

In [ ]:
def flux_ratio(flare_id, band):
    d = SAMPLE / "flares" / flare_id
    flare = json.load(open(d / "flare.json"))
    t_peak = times.to_epoch(flare["peak_utc"])
    st = argex.read(d / f"stack_{band}.fits")
    x, y = st.radec_to_pixel(flare["ra"], flare["dec"])
    flux, _ = aperture_flux(st.data, x, y)
    minutes = (st.epochs - t_peak) / 60
    quiet = np.nanmedian(flux[minutes < -2])
    return minutes, flux / quiet

## 2. Integrate

Integrate $F/F_\mathrm{q} - 1$ over the frames from the start of the flare
on, in seconds (the frames are 60 s apart). Fill a table with one row per
flare and the ED in each band.

In [ ]:
FLARES = sorted(p.name for p in (SAMPLE / "flares").iterdir())
rows = []
fig, axes = plt.subplots(1, len(FLARES), figsize=(12, 3), sharey=True)
for ax, flare_id in zip(axes, FLARES):
    row = {"flare": flare_id}
    for band, color in (("beta", "tab:blue"), ("rho", "tab:red")):
        minutes, ratio = flux_ratio(flare_id, band)
        during = (minutes >= -2) & np.isfinite(ratio)
        row[f"ED_{band}_s"] = np.sum(ratio[during] - 1) * 60.0
        ax.plot(minutes, ratio, "o-", ms=2, color=color, label=GREEK[band])
    row["beta/rho"] = row["ED_beta_s"] / row["ED_rho_s"]
    rows.append(row)
    ax.set(title=flare_id, xlabel="minutes from peak", yscale="log")
axes[0].set_ylabel("F / F_q"); axes[0].legend()
plt.show()
eds = Table(rows=rows)
for col in eds.colnames[1:]:
    eds[col].format = ".0f" if col.startswith("ED") else ".1f"
eds